In [1]:
!hdfs dfs -mkdir -p /data

In [4]:
!hdfs dfs -put C:\Users\vongo\Final_BigData\flights.csv /data/
!hdfs dfs -put C:\Users\vongo\Final_BigData\airlines.csv /data/
!hdfs dfs -put C:\Users\vongo\Final_BigData\airports.csv /data/

In [5]:
!hdfs dfs -ls /data/

Found 3 items
-rw-r--r--   1 vongo supergroup        359 2026-06-12 00:45 /data/airlines.csv
-rw-r--r--   1 vongo supergroup      23867 2026-06-12 00:45 /data/airports.csv
-rw-r--r--   1 vongo supergroup  592406591 2026-06-12 00:45 /data/flights.csv


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Flight Data Analysis – Spark SQL") \
    .getOrCreate()

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [ ]:
flights_df  = spark.read.csv("hdfs:///data/flights.csv",   header=True, inferSchema=True)
airlines_df = spark.read.csv("hdfs:///data/airlines.csv",  header=True, inferSchema=True)
airports_df = spark.read.csv("hdfs:///data/airports.csv",  header=True, inferSchema=True)

flights_df.createOrReplaceTempView("flights")
airlines_df.createOrReplaceTempView("airlines")
airports_df.createOrReplaceTempView("airports")

# QUERY 1: Systemic Failure Detection – Airports with the Worst On-Time Departure Rate

In [ ]:
query1 = spark.sql("""
    SELECT
        f.ORIGIN_AIRPORT                                                       AS airport_code,
        a.CITY                                                                 AS city,
        a.STATE                                                                AS state,
        COUNT(*)                                                               AS total_flights,
        ROUND(SUM(CASE WHEN f.DEPARTURE_DELAY > 15 THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                                   AS delay_rate_pct,
        ROUND(AVG(f.DEPARTURE_DELAY), 2)                                       AS avg_dep_delay_min,
        ROUND(AVG(f.AIR_SYSTEM_DELAY), 2)                                      AS avg_air_system_delay,
        ROUND(AVG(f.AIRLINE_DELAY), 2)                                         AS avg_airline_delay,
        ROUND(AVG(f.WEATHER_DELAY), 2)                                         AS avg_weather_delay
    FROM flights f
    LEFT JOIN airports a ON f.ORIGIN_AIRPORT = a.IATA_CODE
    WHERE f.CANCELLED = 0
      AND f.DEPARTURE_DELAY IS NOT NULL
    GROUP BY f.ORIGIN_AIRPORT, a.CITY, a.STATE
    HAVING COUNT(*) >= 1000
    ORDER BY delay_rate_pct DESC
    LIMIT 20
""")

print("=== Q1: Airports with Worst On-Time Departure Rate (by Delay Source) ===")
query1.show(truncate=False)

# QUERY 2: Geographical Weather Bottlenecks – States Most Affected by Weather Delay

In [ ]:
query2 = spark.sql("""
    SELECT
        a.STATE                                                                     AS state,
        COUNT(*)                                                                    AS total_departures,
        ROUND(SUM(CASE WHEN f.WEATHER_DELAY > 0 THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                                        AS weather_affected_pct,
        ROUND(AVG(CASE WHEN f.WEATHER_DELAY > 0 THEN f.WEATHER_DELAY END), 2)      AS avg_weather_delay_when_affected,
        SUM(CASE WHEN f.CANCELLED = 1
                  AND f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END)               AS weather_cancellations,
        ROUND(SUM(CASE WHEN f.CANCELLED = 1
                        AND f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END) * 100.0
              / COUNT(*), 2)                                                        AS weather_cancel_rate_pct
    FROM flights f
    LEFT JOIN airports a ON f.ORIGIN_AIRPORT = a.IATA_CODE
    WHERE a.STATE IS NOT NULL
    GROUP BY a.STATE
    HAVING COUNT(*) >= 500
    ORDER BY weather_affected_pct DESC
    LIMIT 20
""")

print("=== Q2: Geographical Weather Bottlenecks by State ===")
query2.show(truncate=False)

# QUERY 3: High-Frequency, High-Cancellation Flight Paths


In [ ]:
query3 = spark.sql("""
    SELECT
        f.ORIGIN_AIRPORT                                                        AS origin,
        ap1.CITY                                                                AS origin_city,
        f.DESTINATION_AIRPORT                                                   AS destination,
        ap2.CITY                                                                AS dest_city,
        COUNT(*)                                                                AS total_flights,
        SUM(f.CANCELLED)                                                        AS total_cancellations,
        ROUND(SUM(f.CANCELLED) * 100.0 / COUNT(*), 2)                          AS cancellation_rate_pct,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'A' THEN 1 ELSE 0 END)           AS cancelled_by_airline,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'B' THEN 1 ELSE 0 END)           AS cancelled_by_weather,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'C' THEN 1 ELSE 0 END)           AS cancelled_by_nas,
        SUM(CASE WHEN f.CANCELLATION_REASON = 'D' THEN 1 ELSE 0 END)           AS cancelled_by_security
    FROM flights f
    LEFT JOIN airports ap1 ON f.ORIGIN_AIRPORT      = ap1.IATA_CODE
    LEFT JOIN airports ap2 ON f.DESTINATION_AIRPORT = ap2.IATA_CODE
    GROUP BY f.ORIGIN_AIRPORT, ap1.CITY, f.DESTINATION_AIRPORT, ap2.CITY
    HAVING COUNT(*) >= 500                        -- chỉ xét route tần suất cao
       AND SUM(f.CANCELLED) * 100.0 / COUNT(*) >= 2.0  -- tỷ lệ hủy >= 2%
    ORDER BY cancellation_rate_pct DESC, total_flights DESC
    LIMIT 20
""")

print("=== Q3: High-Frequency, High-Cancellation Flight Paths ===")
query3.show(truncate=False)

# QUERY 4: Cascading Delay Analysis – Identifying Airlines with Late Aircraft Problem

In [ ]:
query4 = spark.sql("""
    SELECT
        al.AIRLINE                                                                          AS airline_name,
        COUNT(*)                                                                            AS total_delayed_flights,
        ROUND(AVG(f.DEPARTURE_DELAY), 2)                                                    AS avg_total_dep_delay,
        ROUND(AVG(f.LATE_AIRCRAFT_DELAY), 2)                                                AS avg_late_aircraft_delay,
        ROUND(AVG(f.AIRLINE_DELAY), 2)                                                      AS avg_airline_delay,
        ROUND(AVG(f.AIR_SYSTEM_DELAY), 2)                                                   AS avg_air_system_delay,
        ROUND(AVG(f.WEATHER_DELAY), 2)                                                      AS avg_weather_delay,
        ROUND(
            AVG(f.LATE_AIRCRAFT_DELAY) * 100.0 /
            NULLIF(AVG(f.AIR_SYSTEM_DELAY) + AVG(f.AIRLINE_DELAY) +
                   AVG(f.LATE_AIRCRAFT_DELAY) + AVG(f.WEATHER_DELAY) +
                   AVG(f.SECURITY_DELAY), 0),
        2)                                                                                  AS late_aircraft_share_pct
    FROM flights f
    JOIN airlines al ON f.AIRLINE = al.IATA_CODE
    WHERE f.CANCELLED = 0
      AND f.DEPARTURE_DELAY > 15
    GROUP BY al.AIRLINE
    ORDER BY late_aircraft_share_pct DESC
""")

print("=== Q4: Cascading Delay Analysis (Late Aircraft Share per Airline) ===")
query4.show(truncate=False)

# QUERY 5: Monthly Delay Seasonality – Detecting Systemic Seasonal Bottlenecks


In [ ]:
query5 = spark.sql("""
    SELECT
        MONTH,
        COUNT(*)                                                                AS total_flights,
        ROUND(SUM(CANCELLED) * 100.0 / COUNT(*), 2)                            AS cancellation_rate_pct,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN DEPARTURE_DELAY END), 2)        AS avg_dep_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN WEATHER_DELAY END), 2)          AS avg_weather_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN AIR_SYSTEM_DELAY END), 2)       AS avg_air_system_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN AIRLINE_DELAY END), 2)          AS avg_airline_delay,
        ROUND(AVG(CASE WHEN CANCELLED = 0 THEN LATE_AIRCRAFT_DELAY END), 2)    AS avg_late_aircraft_delay
    FROM flights
    GROUP BY MONTH
    ORDER BY MONTH
""")

print("=== Q5: Monthly Delay Seasonality – Systemic Seasonal Bottlenecks ===")
query5.show(truncate=False)
